RUNTIME: T4 or A100     
INSTRUCTIONS:

1. Set runtime to GPU: Runtime → Change runtime type → GPU → T4 or A100
2. Run all cells in order: Runtime → Run all
3. When prompted, authorize Google Drive access
4. Model checkpoints are saved to Google Drive via:
```
python   model.save_pretrained(SAVE_DIR + "checkpoints/setfit_model")
```
Make sure SAVE_DIR is set correctly in the setup cell   

5. To save results/figures to GitHub:   
    a. Right-click the file in the left sidebar → Download  
    b. Go to the GitHub repo → results/ → Add file → Upload files   
    c. Commit directly on GitHub    
6. Save the notebook: Ctrl+S → save to GitHub when prompted

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────
!pip install -q setfit accelerate

# ── 2. Mount Google Drive ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/cs4120_project/"

# ── 3. Standard imports ───────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report
import torch
from setfit import SetFitModel, Trainer, TrainingArguments

# ── 4. Sanity checks ──────────────────────────────────────────────
print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Shared Evaluation Wiring

This section integrates `src/evaluate.py` so SetFit uses the same metrics/report format as other models.
Run this cell once, then call `evaluate_setfit_predictions(...)` after generating `y_pred_binary` for test data.


In [ ]:
import ast
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.preprocessing import MultiLabelBinarizer

# Resolve repo root so src/ imports work in both local Jupyter and Colab
repo_candidates = [Path.cwd(), Path.cwd().parent]
repo_root = None
for candidate in repo_candidates:
    if (candidate / "src" / "evaluate.py").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not find repo root containing src/evaluate.py")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.evaluate import evaluate_run, save_evaluation_outputs

def _parse_labels_column(df, label_col="labels"):
    df = df.copy()

    def _parse_one(value):
        if isinstance(value, list):
            return [int(x) for x in value]
        if isinstance(value, np.ndarray):
            return [int(x) for x in value.tolist()]
        if isinstance(value, str):
            s = value.strip()
            if s.startswith('[') and s.endswith(']'):
                body = s[1:-1].strip()
                if body == "":
                    return []
                if ',' in body:
                    tokens = [t.strip() for t in body.split(',') if t.strip()]
                else:
                    tokens = [t for t in body.split() if t]
                return [int(t) for t in tokens]
            return [int(x) for x in ast.literal_eval(s)]
        return value

    df[label_col] = df[label_col].apply(_parse_one)
    return df

def _resolve_data_dir():
    # Local notebooks usually run from notebooks/, so data is often ../data
    for candidate in [repo_root / "data", Path("../data"), Path("data")]:
        if (candidate / "test.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data directory containing test.csv")

def _get_label_names():
    dataset = load_dataset("google-research-datasets/go_emotions", "simplified")
    return dataset["train"].features["labels"].feature.names

def evaluate_setfit_predictions(
    y_pred_binary,
    *,
    data_fraction,
    seed,
    method="setfit",
    data_dir=None,
    output_dir=None,
):
    """
    Evaluate SetFit predictions on the shared test split and save standard outputs.

    y_pred_binary: binary matrix (n_samples, n_labels) aligned with test.csv rows.
    """
    data_dir = Path(data_dir) if data_dir else _resolve_data_dir()
    output_dir = Path(output_dir) if output_dir else (repo_root / "results")

    test_df = pd.read_csv(data_dir / "test.csv")
    test_df = _parse_labels_column(test_df, label_col="labels")

    label_names = _get_label_names()
    mlb = MultiLabelBinarizer(classes=list(range(len(label_names))))
    mlb.fit([list(range(len(label_names)))])
    y_true_binary = mlb.transform(test_df["labels"])

    evaluation = evaluate_run(
        method=method,
        data_fraction=float(data_fraction),
        seed=int(seed),
        label_names=label_names,
        y_true=y_true_binary,
        y_pred=y_pred_binary,
    )

    paths = save_evaluation_outputs(evaluation, method=method, output_dir=output_dir)
    return evaluation, paths

print("Shared evaluation helpers loaded.")
print("After generating SetFit predictions, call evaluate_setfit_predictions(y_pred_binary, data_fraction=..., seed=...)")
